# Cellfix29 · Development sau sửa trên 9 ca

Notebook Private. Bật GPU T4 x2, Internet On và gắn dataset checkpoint chứa research-rag-kaggle-v2.zip. Chạy Cell 1–4 theo thứ tự.

Lượt này giữ 9 câu và 3 PDF của bộ đã chấm. Runtime có phép kiểm chứng số đếm từ ô bảng và từ đoạn văn. File tham chiếu dùng riêng sau inference.


## 1. Khởi tạo workspace và kiểm checkpoint

In [ ]:
RUNTIME_SHA256 = 'e162edb9ed94bd7ece1eece1dff6277fec5c923ef15015b5a370b13a0a640769'
PAYLOAD_SHA256 = 'c2f6decea7021c83a99a7cb4e4985c4d511723fdebb4c6066a0d97e4523fbaa5'
ASSETS = {'05_writer/checkpoints/writer_qlora/adapter_config.json': {'sha256': 'ca0c0487fcd867c57881b4a820a1fa909964a7799221b242475e05103ca7090d', 'size': 838}, '05_writer/checkpoints/writer_qlora/adapter_model.safetensors': {'sha256': '2c233996b954baa8cb9b19d55e586e3ab59db9beeb7e80325f5d82f69d8d2a27', 'size': 73911112}, '05_writer/checkpoints/writer_qlora/added_tokens.json': {'sha256': '58b54bbe36fc752f79a24a271ef66a0a0830054b4dfad94bde757d851968060b', 'size': 605}, '05_writer/checkpoints/writer_qlora/merges.txt': {'sha256': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5', 'size': 1671853}, '05_writer/checkpoints/writer_qlora/special_tokens_map.json': {'sha256': '76862e765266b85aa9459767e33cbaf13970f327a0e88d1c65846c2ddd3a1ecd', 'size': 613}, '05_writer/checkpoints/writer_qlora/tokenizer.json': {'sha256': '9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa', 'size': 11421896}, '05_writer/checkpoints/writer_qlora/tokenizer_config.json': {'sha256': '57fea41fa99fce34f91450643e48b3bfbb078c32c6831a3c06834f122c200510', 'size': 7336}, '05_writer/checkpoints/writer_qlora/vocab.json': {'sha256': 'ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910', 'size': 2776833}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/adapter_config.json': {'sha256': '449eae6ce9dc58c5c783f99a89242193f18c06b405ab9a86b9a44b3343b6ddc7', 'size': 1054}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/adapter_model.safetensors': {'sha256': 'd3123dcea7298404133b0db359f533ceb2f4334a52ed93dcb11331e5de4d0c06', 'size': 40400200}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/chat_template.jinja': {'sha256': 'cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f', 'size': 2507}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/tokenizer.json': {'sha256': '3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8', 'size': 11421892}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/tokenizer_config.json': {'sha256': '47bfa3e7727312946b29ac10d6dd0672d63cf7815b2a160b9523872040d2e536', 'size': 665}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/adapter_config.json': {'sha256': 'b3f8aaf9cbfba12f590fbe96f22b3acb2c3a647c866ed0f5aef7edcdb15285d7', 'size': 1103}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/adapter_model.safetensors': {'sha256': 'bb844bd730bfb1aa49afaedd6c6ee4c66b5553966eec7c3ea443d32c40352b7c', 'size': 161533192}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/chat_template.jinja': {'sha256': 'cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f', 'size': 2507}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/tokenizer.json': {'sha256': '3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8', 'size': 11421892}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/tokenizer_config.json': {'sha256': '47bfa3e7727312946b29ac10d6dd0672d63cf7815b2a160b9523872040d2e536', 'size': 665}}
RUNTIME_B64 = '...'

from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import base64, hashlib, os, shutil, subprocess, sys, tempfile

subprocess.run(["nvidia-smi", "-L"], check=True)
input_dir = Path("/kaggle/input")
session = Path(tempfile.mkdtemp(prefix="cellfix29-e2e-", dir="/tmp"))
root = session / "workspace"
root.mkdir()
runtime = base64.b64decode(RUNTIME_B64)
assert hashlib.sha256(runtime).hexdigest() == RUNTIME_SHA256, "Gói runtime sai hash"
with ZipFile(BytesIO(runtime)) as bundle:
    bundle.extractall(root)
payload = root / "artifacts/atomic-cellfix29-e2e-quantity-candidate.zip"
assert hashlib.sha256(payload.read_bytes()).hexdigest() == PAYLOAD_SHA256
candidate = root / "research-runs/atomic-cellfix29-candidate/workspace"
with ZipFile(payload) as bundle:
    bundle.extractall(candidate)

anchor = "train_agent3_summarizer/checkpoints/generator_qlora/final/adapter_config.json"
found = []
for path in input_dir.rglob("adapter_config.json"):
    if path.as_posix().endswith("/" + anchor):
        top = path.parents[len(Path(anchor).parts) - 1]
        if all((top / rel).is_file() for rel in ASSETS):
            found.append(top)
assert len(found) <= 1, "Có nhiều workspace checkpoint. Chỉ giữ một input phù hợp."
def verify(path, expected):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    assert path.stat().st_size == expected["size"] and digest.hexdigest() == expected["sha256"], path

if found:
    print("Checkpoint input", found[0])
    for rel, expected in ASSETS.items():
        source = found[0] / rel
        verify(source, expected)
        target = root / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
else:
    matches = list(input_dir.rglob("research-rag-kaggle-v2.zip"))
    assert len(matches) == 1, "Thêm dataset Private research-rag-five-agents-workspace hoặc gói v2 đúng phiên bản."
    print("Checkpoint archive", matches[0])
    with ZipFile(matches[0]) as checkpoint_zip:
        for rel, expected in ASSETS.items():
            target = root / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            with checkpoint_zip.open(rel) as source, target.open("wb") as out:
                shutil.copyfileobj(source, out)
            verify(target, expected)

print("Workspace", root)
print("Checkpoint và runtime đã khớp hash")
print("Disk /tmp còn trống (GiB)", round(shutil.disk_usage("/tmp").free / 2**30, 1))


## 2. Dùng lại CUDA/PyTorch của Kaggle và chạy preflight

Chờ **CELL 2 PASS**. Nếu lỗi, tải thư mục `cellfix29-cell2-*` trong Output để kiểm log.


In [ ]:
# Paste into Cell 2 after running Cell 1 of the Cellfix29 E2E notebook.
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

setup_dir = Path(tempfile.mkdtemp(prefix="cellfix29-cell2-", dir="/kaggle/working"))
log_path = setup_dir / "cell2-setup.log"
status_path = setup_dir / "cell2-status.json"
venv = Path(tempfile.mkdtemp(prefix="venv-kaggle-", dir=session))
python = str(venv / "bin/python")
env = {**os.environ, "PYTHONPATH": str(root), "HF_HOME": "/tmp/cellfix29-e2e-hf"}


def setup_status(stage, state, **details):
    status_path.write_text(json.dumps({
        "stage": stage, "state": state,
        "tmp_free_gib": round(shutil.disk_usage(session).free / 2**30, 2),
        **details,
    }, indent=2), encoding="utf-8")


def run_setup(stage, command):
    setup_status(stage, "running")
    print(f"Cell 2 · {stage}", flush=True)
    with log_path.open("a", encoding="utf-8") as log:
        print(f"\n=== {stage} ===", file=log, flush=True)
        completed = subprocess.run(command, cwd=root, env=env, stdout=log, stderr=subprocess.STDOUT)
    if completed.returncode:
        setup_status(stage, "failed", exit_code=completed.returncode)
        print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-18:]))
        raise RuntimeError(f"{stage} exited {completed.returncode}. See {log_path}")
    setup_status(stage, "completed")


print("Cell 2 setup files", setup_dir, flush=True)
run_setup("gpu", ["nvidia-smi"])
run_setup("reuse-kaggle-python", [
    sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(venv),
])

base_info = setup_dir / "cuda-base.json"
probe = """
from importlib import metadata
import json, re, sys
from pathlib import Path
import torch

assert torch.version.cuda and torch.cuda.is_available(), "Kaggle PyTorch cannot access CUDA"
assert torch.ones(1, device="cuda:0").item() == 1
names = {re.sub(r"[-_.]+", "-", d.metadata["Name"]).lower()
         for d in metadata.distributions() if d.metadata.get("Name")}
protected = {n: metadata.version(n) for n in sorted(names)
             if n in {"torch", "torchvision", "torchaudio", "triton", "pytorch-triton"}
             or n.startswith(("nvidia-", "cuda-"))}
info = dict(python=sys.version, torch=torch.__version__, cuda=torch.version.cuda,
            torch_path=str(Path(torch.__file__).resolve()), packages=protected,
            gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
Path(sys.argv[1]).write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))
"""
run_setup("base-cuda-check", [python, "-u", "-c", probe, str(base_info)])
base = json.loads(base_info.read_text())
print(f"Reusing Kaggle torch {base['torch']} · CUDA {base['cuda']}", flush=True)

constraints = setup_dir / "cuda-constraints.txt"
constraints.write_text("".join(f"{name}=={version}\n" for name, version in base["packages"].items()))
report = setup_dir / "pip-plan.json"
pip = [python, "-m", "pip", "--isolated", "install", "--disable-pip-version-check",
       "--progress-bar", "off", "--only-binary=:all:"]
run_setup("resolve-project-dependencies", [
    *pip, "--dry-run", "--report", str(report), "-c", str(constraints),
    "-r", "requirements-orchestrator.txt", "-r", "requirements-atomic-verifier.txt",
])
planned = json.loads(report.read_text())["install"]
import re

names = {re.sub(r"[-_.]+", "-", item["metadata"]["name"]).lower() for item in planned}
blocked = {n for n in names if n in base["packages"]
           or n in {"torch", "torchvision", "torchaudio", "triton", "pytorch-triton"}
           or n.startswith(("nvidia-", "cuda-"))}
if blocked:
    setup_status("resolve-project-dependencies", "blocked", protected_packages=sorted(blocked))
    raise RuntimeError(f"Install plan touches the Kaggle CUDA stack: {sorted(blocked)}. See {report}")
if planned:
    pins = [f"{item['metadata']['name']}=={item['metadata']['version']}" for item in planned]
    # The dry run resolves transitive dependencies. Install only that approved package list.
    run_setup("install-approved-packages", [*pip, "--no-deps", *pins])

preflight = """
from importlib import metadata
import json, sys
from pathlib import Path
from packaging.requirements import Requirement
import torch
from bitsandbytes.nn import Linear4bit
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import faiss
from research_rag.atomic_adapter import cellfix29_runtime
from research_rag.adapters import PdfReader

expected = json.loads(Path(sys.argv[1]).read_text())
assert str(Path(torch.__file__).resolve()) == expected["torch_path"]
assert torch.__version__ == expected["torch"] and torch.version.cuda == expected["cuda"]
assert all(metadata.version(name) == version for name, version in expected["packages"].items())
for file in ("requirements-orchestrator.txt", "requirements-atomic-verifier.txt"):
    for line in Path(file).read_text().splitlines():
        spec = line.split("#", 1)[0].strip()
        if spec:
            req = Requirement(spec)
            version = metadata.version(req.name)
            assert req.specifier.contains(version), f"{req.name}={version} differs from {spec}"

with torch.inference_mode():
    layer = Linear4bit(64, 64, bias=False, compute_dtype=torch.float16, quant_type="nf4").to("cuda:0")
    x = torch.ones((1, 64), device="cuda:0", dtype=torch.float16)
    assert torch.isfinite(layer(x)).all().item(), "NF4 forward failed"
runtime = cellfix29_runtime()
assert runtime.atomic.POLICY == "atomic-verifier-constrained-v8-cellfix29"
chunks = PdfReader().read("research_rag/evaluation/pdfs/simple_text.pdf")
assert any("The Aster trial enrolled 48 participants." in " ".join(c["text"].split())
           and c["metadata"]["page_no"] == 0 for c in chunks)
print("CUDA and NF4 passed")
print("Policy", runtime.atomic.POLICY)
print("PDF evidence is present on the first page")
"""
run_setup("cuda-nf4-cellfix29-pdf-preflight", [python, "-u", "-c", preflight, str(base_info)])
setup_status("cell2", "passed", torch=base["torch"], cuda=base["cuda"])
print("CELL 2 PASS · CUDA/NF4, Cellfix29 và PDF. Có thể chạy Cell 3.")


## 3. Chạy chín câu đã khóa

Các ca chạy độc lập theo thứ tự. Mỗi ca giữ stdout, stderr, trạng thái tiến trình và trace nếu có. Lượt đầu có thể tải model vào cache.

In [ ]:
INPUTS_JSON = '[\n  {\n    "id": "viisa-text",\n    "paper": "viisa",\n    "question": "How many annotation rounds does the ViISA paper report?"\n  },\n  {\n    "id": "viisa-table",\n    "paper": "viisa",\n    "question": "According to Table 1, how many negative samples and how many positive samples are in ViISA, respectively?"\n  },\n  {\n    "id": "viisa-missing",\n    "paper": "viisa",\n    "question": "What exact decoding temperature does the ViISA paper report for the zero-shot GPT-4o result in Table 1?"\n  },\n  {\n    "id": "visp-text",\n    "paper": "visp",\n    "question": "According to the abstract, approximately how many original–paraphrase pairs are in ViSP?"\n  },\n  {\n    "id": "visp-table",\n    "paper": "visp",\n    "question": "According to the Sentence Pair row of Table 4, how many total sentence pairs are in the Train and Test sets of ViSP, respectively?"\n  },\n  {\n    "id": "visp-missing",\n    "paper": "visp",\n    "question": "What exact random seed does the ViSP paper report for selecting the 200 samples per split for human evaluation?"\n  },\n  {\n    "id": "triple-sets-text",\n    "paper": "triple-sets",\n    "question": "How many LLMs does the abstract say were benchmarked on the new long-input dataset?"\n  },\n  {\n    "id": "triple-sets-table",\n    "paper": "triple-sets",\n    "question": "In Table 1, what AvgLenW does GPT4.1 obtain in English and in Irish, respectively?"\n  },\n  {\n    "id": "triple-sets-missing",\n    "paper": "triple-sets",\n    "question": "What exact random seed is reported for the random sampling of twenty-one data points used to produce Hum-1.0?"\n  }\n]\n'
INPUTS_SHA256 = 'a2c9b95fa14b0841c3e5eb18f4a36886c1d381b9ada38ef8a9f0eb2df1ee031d'
LOCK_JSON = '{\n  "candidate": "cellfix29-e2e-structured-counts",\n  "evaluation": "cross-paper-development-v1",\n  "qaspar_reference_run": {\n    "artifact": "cellfix29-e2e-qasper-grounded-claims-nod6pgb1.zip",\n    "sha256": "eb19f3999ddf3c1ed7cb187e43bd0b3471ff1e714e035c98869f5d08a68c0834",\n    "observed_smoke": "3/3",\n    "pdf_sha256": "b15b4378988ed4e553dd312eaf334411abcf1fb28c53b846d360435eb55cd193"\n  },\n  "base_notebook": "kaggle_cellfix29_e2e_cross_paper.ipynb",\n  "base_notebook_sha256": "358a267d8ce731efcb968bf69eb15b0c80f810efbba91fde7b5a0c730b6690f8",\n  "runtime_sha256": "e162edb9ed94bd7ece1eece1dff6277fec5c923ef15015b5a370b13a0a640769",\n  "payload_sha256": "c2f6decea7021c83a99a7cb4e4985c4d511723fdebb4c6066a0d97e4523fbaa5",\n  "cellfix29_baseline_payload_sha256": "83c4c54562d3a68049328b635dc30d1feac5fa0409f9dfe234d3cd0f6fd1baa0",\n  "checkpoint_files": {\n    "05_writer/checkpoints/writer_qlora/adapter_config.json": {\n      "sha256": "ca0c0487fcd867c57881b4a820a1fa909964a7799221b242475e05103ca7090d",\n      "size": 838\n    },\n    "05_writer/checkpoints/writer_qlora/adapter_model.safetensors": {\n      "sha256": "2c233996b954baa8cb9b19d55e586e3ab59db9beeb7e80325f5d82f69d8d2a27",\n      "size": 73911112\n    },\n    "05_writer/checkpoints/writer_qlora/added_tokens.json": {\n      "sha256": "58b54bbe36fc752f79a24a271ef66a0a0830054b4dfad94bde757d851968060b",\n      "size": 605\n    },\n    "05_writer/checkpoints/writer_qlora/merges.txt": {\n      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5",\n      "size": 1671853\n    },\n    "05_writer/checkpoints/writer_qlora/special_tokens_map.json": {\n      "sha256": "76862e765266b85aa9459767e33cbaf13970f327a0e88d1c65846c2ddd3a1ecd",\n      "size": 613\n    },\n    "05_writer/checkpoints/writer_qlora/tokenizer.json": {\n      "sha256": "9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa",\n      "size": 11421896\n    },\n    "05_writer/checkpoints/writer_qlora/tokenizer_config.json": {\n      "sha256": "57fea41fa99fce34f91450643e48b3bfbb078c32c6831a3c06834f122c200510",\n      "size": 7336\n    },\n    "05_writer/checkpoints/writer_qlora/vocab.json": {\n      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910",\n      "size": 2776833\n    },\n    "train_agent3_summarizer/checkpoints/critic_qlora/final/adapter_config.json": {\n      "sha256": "449eae6ce9dc58c5c783f99a89242193f18c06b405ab9a86b9a44b3343b6ddc7",\n      "size": 1054\n    },\n    "train_agent3_summarizer/checkpoints/critic_qlora/final/adapter_model.safetensors": {\n      "sha256": "d3123dcea7298404133b0db359f533ceb2f4334a52ed93dcb11331e5de4d0c06",\n      "size": 40400200\n    },\n    "train_agent3_summarizer/checkpoints/critic_qlora/final/chat_template.jinja": {\n      "sha256": "cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f",\n      "size": 2507\n    },\n    "train_agent3_summarizer/checkpoints/critic_qlora/final/tokenizer.json": {\n      "sha256": "3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8",\n      "size": 11421892\n    },\n    "train_agent3_summarizer/checkpoints/critic_qlora/final/tokenizer_config.json": {\n      "sha256": "47bfa3e7727312946b29ac10d6dd0672d63cf7815b2a160b9523872040d2e536",\n      "size": 665\n    },\n    "train_agent3_summarizer/checkpoints/generator_qlora/final/adapter_config.json": {\n      "sha256": "b3f8aaf9cbfba12f590fbe96f22b3acb2c3a647c866ed0f5aef7edcdb15285d7",\n      "size": 1103\n    },\n    "train_agent3_summarizer/checkpoints/generator_qlora/final/adapter_model.safetensors": {\n      "sha256": "bb844bd730bfb1aa49afaedd6c6ee4c66b5553966eec7c3ea443d32c40352b7c",\n      "size": 161533192\n    },\n    "train_agent3_summarizer/checkpoints/generator_qlora/final/chat_template.jinja": {\n      "sha256": "cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f",\n      "size": 2507\n    },\n    "train_agent3_summarizer/checkpoints/generator_qlora/final/tokenizer.json": {\n      "sha256": "3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8",\n      "size": 11421892\n    },\n    "train_agent3_summarizer/checkpoints/generator_qlora/final/tokenizer_config.json": {\n      "sha256": "47bfa3e7727312946b29ac10d6dd0672d63cf7815b2a160b9523872040d2e536",\n      "size": 665\n    }\n  },\n  "checkpoints_origin": "Per-file hashes enforced by the verified notebook preflight.",\n  "candidate_runtime_config": {\n    "gpu": "T4 x2",\n    "retrieval": "dense",\n    "initial_top_k": 5,\n    "verifier_arg": "atomic",\n    "quantize_4bit": true,\n    "process_timeout": 3600,\n    "model_dependencies": "Inherited from the locked embedded runtime."\n  },\n  "inference_inputs_sha256": "a2c9b95fa14b0841c3e5eb18f4a36886c1d381b9ada38ef8a9f0eb2df1ee031d",\n  "references_sha256": "0031242bfe09ca9a1f24e722bc551dd98f149492cd32fb179eb036484183bd4a",\n  "authoring_manifest_sha256": "f99e532af932effe1ab26d3048e7785c5cace13dbee631b9d08d2f79e816cccb",\n  "papers": {\n    "viisa": {\n      "anthology_id": "2025.inlg-main.14",\n      "title": "Exploring the Power of Large Language Models for Vietnamese Implicit Sentiment Analysis",\n      "authors": [\n        "Huy Gia Luu",\n        "Dang Van Thin"\n      ],\n      "file": "2025.inlg-main.14.pdf",\n      "pages": 13,\n      "sha256": "57dc3d03781b4f3987e2bcbb5816fe3b2ad40bc8a0177cffda6379890d27904b",\n      "pdf_url": "https://aclanthology.org/2025.inlg-main.14.pdf",\n      "license": "CC BY 4.0",\n      "license_source": "https://aclanthology.org/faq/#what-is-the-copyright-for-materials-in-the-acl-anthology"\n    },\n    "visp": {\n      "anthology_id": "2025.naacl-findings.59",\n      "title": "A Large-Scale Benchmark for Vietnamese Sentence Paraphrases",\n      "authors": [\n        "Sang Quang Nguyen",\n        "Kiet Van Nguyen"\n      ],\n      "file": "2025.naacl-findings.59.pdf",\n      "pages": 16,\n      "sha256": "805b329011d77b96801df4f78e210d076c5393488c2f838bf76474051624c413",\n      "pdf_url": "https://aclanthology.org/2025.naacl-findings.59.pdf",\n      "license": "CC BY 4.0",\n      "license_source": "https://aclanthology.org/faq/#what-is-the-copyright-for-materials-in-the-acl-anthology"\n    },\n    "triple-sets": {\n      "anthology_id": "2025.inlg-main.47",\n      "title": "Scaling Up Data-to-Text Generation to Longer Sequences: A New Dataset and Benchmark Results for Generation from Large Triple Sets",\n      "authors": [\n        "Chinonso Cynthia Osuji",\n        "Simon Mille",\n        "Ornait O’Connell",\n        "Thiago Castro Ferreira",\n        "Anya Belz",\n        "Brian Davis"\n      ],\n      "file": "2025.inlg-main.47.pdf",\n      "pages": 13,\n      "sha256": "b3811bdde740f2bb89fb8d41b9a056ed3d04f18e935e2cab4b8e570d2abbf2a8",\n      "pdf_url": "https://aclanthology.org/2025.inlg-main.47.pdf",\n      "license": "CC BY 4.0",\n      "license_source": "https://aclanthology.org/faq/#what-is-the-copyright-for-materials-in-the-acl-anthology"\n    }\n  },\n  "parent_run": {\n    "artifact": "cellfix29-e2e-cross-paper-v1-0i9ajoct.zip",\n    "sha256": "daee61729bf9945f38661d52829b3d8a91dd58ad7fb80f0f36e8cc08e68edad4",\n    "lock_sha256": "53b84ce169da1b316e954f80bbbf41d755c29e90581e5af7f7fe1b1dbf298c8f",\n    "observed_cross_paper": "3/9"\n  },\n  "embedded_module_sha256": {\n    "research_rag/atomic_adapter.py": "8bb79efd37bc86249cc1ace11c35bfa19af84e28ee114fdcdaed37d943ca48d3",\n    "research_rag/graph.py": "2f48cb289f756c8e918afeae668e7c52e5994242996fcb204d7537d378cb03ad",\n    "research_rag/state.py": "1eb734656404ded18dc006d79d201fc0b64337026af9b93e692d891d54e41eb1",\n    "research_rag/pdf_count_evidence.py": "7e56f4b36fc6e0fd2d4d835d2b8eb13bd7c295a49bf7e0645a332137b3c7f4a3",\n    "research_rag/pdf_table_cells.py": "5aeed483161556f66bb4b9b67ed462963cf6dc295d720e6a29965b5a6a95286e",\n    "train_agent3_summarizer/node.py": "d09d3e3b441e737ee8a83f96539817f69d7fd20683af379849ec4b5a53ddb957",\n    "train_agent3_summarizer/source_claims.py": "32b6974a94352f8aadf6505637f0f3b8e9498460e4f2ad377ea50d6927a9b01a"\n  }\n}\n'
LOCK_SHA256 = 'b1170df8c1fec0b2d0d4979899b8b6eec5061cf4e771342e60cc50957e614b2a'
PDF_B64 = {'viisa': '...', 'visp': '...', 'triple-sets': '...'}
import json

assert hashlib.sha256(INPUTS_JSON.encode("utf-8")).hexdigest() == INPUTS_SHA256
assert hashlib.sha256(LOCK_JSON.encode("utf-8")).hexdigest() == LOCK_SHA256
inputs = json.loads(INPUTS_JSON)
lock = json.loads(LOCK_JSON)
assert lock["runtime_sha256"] == RUNTIME_SHA256 and lock["payload_sha256"] == PAYLOAD_SHA256
assert lock["inference_inputs_sha256"] == INPUTS_SHA256
case_ids = [case["id"] for case in inputs]
assert len(case_ids) == len(set(case_ids)) == 9
assert set(PDF_B64) == set(lock["papers"]) == {case["paper"] for case in inputs}
pdf_paths = {}
for paper, info in lock["papers"].items():
    content = base64.b64decode(PDF_B64[paper], validate=True)
    assert hashlib.sha256(content).hexdigest() == info["sha256"], paper
    pdf = root / "research_rag/evaluation/pdfs" / info["file"]
    pdf.write_bytes(content)
    pdf_paths[paper] = pdf

out = Path(tempfile.mkdtemp(prefix="cellfix29-e2e-cross-paper-development-v1-", dir="/kaggle/working"))
(out / "candidate-lock.json").write_bytes(LOCK_JSON.encode("utf-8"))
(out / "inference-inputs.json").write_bytes(INPUTS_JSON.encode("utf-8"))
print("Thư mục kết quả", out, flush=True)

for case in inputs:
    case_id, question = case["id"], case["question"]
    pdf = pdf_paths[case["paper"]]
    case_dir = out / case_id
    case_dir.mkdir()
    command = [
        python, "-u", "-m", "research_rag.local",
        "--verifier", "atomic",
        "--question", question, "--pdf", str(pdf),
        "--retrieval-policy", "dense", "--initial-top-k", "5",
        "--quantize-4bit", "--online-model-downloads", "--profile-resources",
        "--process-timeout", "3600",
        "--metrics-output", str(case_dir / "metrics.json"),
        "--events-output", str(case_dir / "events.jsonl"),
        "--trace-output", str(case_dir / "trace.private.jsonl"),
    ]
    print("Đang chạy", case_id, flush=True)
    process = {"state": "interrupted", "exit_code": None}
    with (case_dir / "stdout.log").open("w", encoding="utf-8") as stdout, (
        case_dir / "stderr.log"
    ).open("w", encoding="utf-8") as stderr:
        try:
            completed = subprocess.run(command, cwd=root, env=env, stdout=stdout, stderr=stderr)
            process = {"state": "finished", "exit_code": completed.returncode}
        except OSError as exc:
            process = {"state": "launch_error", "exit_code": None, "error_type": type(exc).__name__}
            print(str(exc), file=stderr)
        finally:
            record = case_dir / "process.json"
            pending = record.with_suffix(".tmp")
            pending.write_text(json.dumps(process), encoding="utf-8")
            pending.replace(record)
    print(case_id, process, flush=True)

## 4. Đóng gói để chấm

Tải ZIP kết quả từ Output. Nếu ZIP có hậu tố -incomplete, giữ lại để chẩn đoán. ZIP chứa trace riêng tư.

In [ ]:
from collections import Counter

def read_json(path, errors):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError) as exc:
        errors.append({"file": path.name, "error_type": type(exc).__name__})
        return {}

def read_jsonl(path, errors):
    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except (OSError, UnicodeError) as exc:
        errors.append({"file": path.name, "error_type": type(exc).__name__})
        return []
    records = []
    for index, line in enumerate(lines, 1):
        try:
            row = json.loads(line)
            if not isinstance(row, dict):
                raise ValueError("Expected an object")
            records.append(row)
        except ValueError as exc:
            errors.append({"file": path.name, "line": index, "error_type": type(exc).__name__})
    return records

review = {
    "candidate_lock_sha256": LOCK_SHA256,
    "runtime_sha256": RUNTIME_SHA256,
    "payload_sha256": PAYLOAD_SHA256,
    "reference_sha256": lock["references_sha256"],
    "inference_inputs_sha256": INPUTS_SHA256,
    "semantic_review": "pending",
    "evaluation": lock["evaluation"],
    "denominator": 9,
    "cases": [],
}
for case in inputs:
    folder = out / case["id"]
    errors = []
    process = read_json(folder / "process.json", errors)
    trace = read_jsonl(folder / "trace.private.jsonl", errors)
    events = read_jsonl(folder / "events.jsonl", errors)
    metrics = read_json(folder / "metrics.json", errors)
    state = next((e["state"] for e in reversed(trace) if e.get("event") == "graph_result"), {})
    calls = Counter(e["component"] for e in events if e.get("event") == "backend_call" and e.get("status") == "ok")
    chunks = {c["source_id"]: c for c in state.get("pdf_chunks", []) if c.get("source_id")}
    citations = []
    for claim in state.get("verified_claims", []):
        chunk = chunks.get(claim["source_id"], {})
        page = chunk.get("metadata", {}).get("page_no")
        citations.append({"claim_id": claim["claim_id"], "source_id": claim["source_id"],
                          "pdf_page": page + 1 if isinstance(page, int) else None,
                          "evidence": claim["evidence"], "source_text": chunk.get("text")})
    review["cases"].append({
        **case,
        "pdf_sha256": lock["papers"][case["paper"]]["sha256"],
        "process": process,
        "graph_status": state.get("status"),
        "stop_reason": state.get("stop_reason"),
        "report": state.get("final_report"),
        "citations": citations,
        "verified_claims": state.get("verified_claims", []),
        "evidence_refs": state.get("evidence_refs", []),
        "backend_calls_ok": dict(calls),
        "table_cell_verifications": [e for e in trace if e.get("event") == "table_cell_verification"],
        "pdf_count_verifications": [e for e in trace if e.get("event") == "pdf_count_verification"],
        "writer_diagnostics": state.get("writer_diagnostics", {}),
        "factcheck_calls_used": state.get("factcheck_calls_used", 0),
        "total_ms": metrics.get("total_ms"),
        "errors": state.get("errors", []),
        "artifact_errors": errors,
    })
complete = (len(review["cases"]) == 9 and
            all(row["process"].get("state") in {"finished", "launch_error"} for row in review["cases"]))
review["complete_roster"] = complete
review["partial_cases"] = [row["id"] for row in review["cases"] if row["graph_status"] == "partial"]
(out / "review.json").write_text(json.dumps(review, ensure_ascii=False, indent=2), encoding="utf-8")
archive = out.with_name(out.name + (".zip" if complete else "-incomplete.zip"))
temporary_archive = archive.with_suffix(".zip.tmp")
with ZipFile(temporary_archive, "w", compression=8) as bundle:
    for file in sorted(out.rglob("*")):
        if file.is_file():
            bundle.write(file, arcname=file.relative_to(out).as_posix())
temporary_archive.replace(archive)
print("ZIP để tải trong mục Output", archive)
if not complete:
    raise RuntimeError("Thiếu ít nhất một kết quả tiến trình. ZIP này là bản chẩn đoán.")